# Stage 04 & 05: Multi-Feature ML Classification & Model Benchmarking
**Project**: Steam Game Intelligence  
**Notebook**: `notebooks/03_nlp_and_machine_learning.ipynb`  
**Objective**: Build a multi-feature ML recommendation pipeline combining TF-IDF review text features with engagement metrics (`hours_played_clean`, `helpful_clean`, `review_word_count`), comparing Logistic Regression vs XGBoost with GridSearchCV tuning.

---
## Pipeline Architecture & Benchmark Metrics:
1. **Target**: `is_recommended` (1 = Recommended, 0 = Not Recommended).
2. **Feature Fusion**: `ColumnTransformer` joining text n-grams with scaled numerical engagement metrics.
3. **Model Benchmark**: Evaluated Logistic Regression vs Tuned XGBoost Classifier.
4. **Leakage Prevention**: All transformations fit strictly on `X_train`.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score

sns.set_theme(style='whitegrid')

# Load Processed Dataset
rev_path = '../data/processed/steam_game_reviews_clean.csv' if os.path.exists('../data/processed/steam_game_reviews_clean.csv') else 'data/processed/steam_game_reviews_clean.csv'
df_reviews = pd.read_csv(rev_path, usecols=['review', 'hours_played_clean', 'helpful_clean', 'review_word_count', 'review_char_len', 'is_recommended'])
df_reviews['review'] = df_reviews['review'].fillna('')
df_reviews = df_reviews[df_reviews['review'].str.strip() != ''].copy()

df_sample = df_reviews.sample(n=100000, random_state=42).reset_index(drop=True)
feature_cols = ['review', 'hours_played_clean', 'helpful_clean', 'review_word_count', 'review_char_len']
X = df_sample[feature_cols]
y = df_sample['is_recommended']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# ColumnTransformer Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(max_features=10000, ngram_range=(1, 2), stop_words='english'), 'review'),
        ('num', StandardScaler(), ['hours_played_clean', 'helpful_clean', 'review_word_count', 'review_char_len'])
    ]
)

# Fit Fine-Tuned XGBoost Pipeline
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss'))
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
y_proba = pipe.predict_proba(X_test)[:, 1]

print("=== Fine-Tuned XGBoost Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['Not Recommended', 'Recommended']))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")
